# 05 — Cross-validation experiments

## Objective
Run the frozen protocol on the 68 development tracks only. The 12-track final holdout is not evaluated. All new output paths are in English.

Run this notebook from your project directory, with the source CSV and the protocol JSON files somewhere below it. File discovery checks contents and the frozen SHA-256 rather than relying on their old names. Set PROJECT_DIR explicitly if necessary. Do not keep unrelated project copies below this directory.

Dependencies: pandas >= 2.0, numpy, scikit-learn >= 1.2, matplotlib, joblib. Restart the kernel and run all cells in order.

This stage produces repeated cross-validation results, not final test results or a confirmed statistical winner. Repeated folds and random seeds are not independent samples.

All cross-validation models are saved locally. The random forest candidate averages predictions from the three frozen seeds on the log scale. Allow the loop to finish; rerunning it overwrites the same experiment outputs.

In [ ]:
from pathlib import Path
import hashlib
import json
import time
import platform
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path.cwd()  # Change only if your input files are elsewhere.
OUTPUT_DIR = PROJECT_DIR / "cross_validation_results"
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Recognize JSON inputs by schema, allowing renamed files/folders.
protocol_candidates = []
fold_candidates = []
for path in PROJECT_DIR.rglob("*.json"):
    if OUTPUT_DIR in path.parents or ".ipynb_checkpoints" in path.parts:
        continue
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except (ValueError, UnicodeError, OSError):
        continue
    if isinstance(value, dict) and {"source_sha256", "features", "target", "split_seeds", "test_family", "rf_params"} <= value.keys():
        protocol_candidates.append((path, value))
    if isinstance(value, list) and value and isinstance(value[0], dict) and {"split_seed", "fold", "train_ids", "validation_ids"} <= value[0].keys():
        fold_candidates.append((path, value))
assert len(protocol_candidates) == 1, "Keep one protocol copy under PROJECT_DIR, or narrow PROJECT_DIR."
protocol_path, protocol = protocol_candidates[0]
matching_folds = [(p, v) for p, v in fold_candidates if p.parent == protocol_path.parent]
assert len(matching_folds) == 1, "Keep the folds JSON beside the protocol JSON."
fold_path, folds = matching_folds[0]
source_candidates = [p for p in PROJECT_DIR.rglob("*.csv")
                     if OUTPUT_DIR not in p.parents
                     and hashlib.sha256(p.read_bytes()).hexdigest() == protocol["source_sha256"]]
assert source_candidates, "No CSV matches the frozen hash. Check the original file; do not change the hash to bypass this check."
source = sorted(source_candidates, key=lambda p: str(p))[0]
FEATURES = protocol["features"]
TARGET = protocol["target"]

# Select development rows before numeric target conversion or any fit.
raw = pd.read_csv(source, sep=";")
raw["album_family"] = raw["album"].replace({"emails i can’t send fwd": "emails i can’t send"})
is_holdout = raw["album_family"].eq(protocol["test_family"])
holdout_ids = set(raw.loc[is_holdout, "track_id"])
dev = raw.loc[~is_holdout].copy()
del raw
for col in [TARGET]:
    dev[col] = pd.to_numeric(dev[col].astype(str).str.replace(".", "", regex=False), errors="raise")
dev["instrumentalness"] = pd.to_numeric(dev["instrumentalness"].astype(str).str.replace(",", ".", regex=False), errors="raise")
dev["duration_min"] = dev["duration_ms"] / 60000
dev = dev.set_index("track_id", verify_integrity=True)
assert len(dev) == protocol["n_development"] == 68
assert len(holdout_ids) == protocol["n_test"] == 12
assert dev["album_family"].nunique() == 6
assert set(dev.index).isdisjoint(holdout_ids)
assert np.isfinite(dev[TARGET]).all() and (dev[TARGET] >= 0).all()
assert all(pd.api.types.is_numeric_dtype(dev[c]) for c in FEATURES)
assert not np.isinf(dev[FEATURES].to_numpy(dtype=float)).any()
assert len(folds) == len(protocol["split_seeds"]) * protocol["folds_per_repeat"]
for row in folds:
    train_ids, val_ids = set(row["train_ids"]), set(row["validation_ids"])
    assert train_ids.isdisjoint(val_ids)
    assert train_ids | val_ids == set(dev.index)
    assert (train_ids | val_ids).isdisjoint(holdout_ids)
    assert set(dev.loc[list(train_ids), "album_family"]).isdisjoint(set(dev.loc[list(val_ids), "album_family"]))
print("Input files recognized despite possible renaming.")
print("Development: 68 tracks, 6 families. Final holdout: excluded.")

## Fixed pipelines

In [ ]:
def make_estimator(name, seed=None):
    steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
    if name == "baseline":
        steps += [("model", DummyRegressor(strategy="mean"))]
    elif name == "linear":
        steps += [("scaler", StandardScaler()), ("model", LinearRegression())]
    elif name == "random_forest":
        steps += [("model", RandomForestRegressor(**protocol["rf_params"], random_state=seed, n_jobs=-1))]
    else:
        raise ValueError(name)
    return TransformedTargetRegressor(regressor=Pipeline(steps), func=np.log1p, inverse_func=np.expm1)

prediction_blocks = []
seed_prediction_blocks = []
fit_logs = []

def prediction_frame(data, prediction_log, model_name, split_seed, fold, partition):
    prediction_log = np.asarray(prediction_log)
    prediction = np.expm1(prediction_log)
    assert prediction.shape == (len(data),)
    assert np.isfinite(prediction).all() and np.isfinite(prediction_log).all()
    assert (prediction >= 0).all(), "Negative prediction: stop and investigate before changing protocol."
    return pd.DataFrame({
        "track_id": data.index.to_numpy(), "album_family": data["album_family"].to_numpy(),
        "y_true": data[TARGET].to_numpy(), "y_true_log": np.log1p(data[TARGET].to_numpy()),
        "y_pred": prediction, "y_pred_log": prediction_log,
        "model": model_name, "split_seed": split_seed, "fold": fold, "partition": partition
    })

for row in folds:
    split_seed, fold = row["split_seed"], row["fold"]
    train, val = dev.loc[row["train_ids"]], dev.loc[row["validation_ids"]]
    X_train, y_train = train[FEATURES], train[TARGET]
    prefix = f"split_{split_seed}_fold_{fold}"
    forest_log_predictions = {"train": [], "validation": []}
    for model_name, model_seed in [("baseline", None), ("linear", None)] + [("random_forest", s) for s in protocol["rf_seeds"]]:
        start = time.perf_counter()
        estimator = make_estimator(model_name, model_seed).fit(X_train, y_train)
        transformed = estimator.regressor_[:-1].transform(X_train)
        assert np.isfinite(transformed).all()
        suffix = model_name if model_seed is None else f"{model_name}_seed_{model_seed}"
        model_path = MODEL_DIR / f"{prefix}_{suffix}.joblib"
        joblib.dump(estimator, model_path)
        for partition, data in [("train", train), ("validation", val)]:
            pred_log = estimator.regressor_.predict(data[FEATURES])
            block = prediction_frame(data, pred_log, model_name, split_seed, fold, partition)
            if model_name == "random_forest":
                forest_log_predictions[partition].append(pred_log)
                block["model_seed"] = model_seed
                seed_prediction_blocks.append(block)
            else:
                prediction_blocks.append(block)
        fit_logs.append({"split_seed": split_seed, "fold": fold, "model": model_name,
                         "model_seed": model_seed, "n_train": len(train), "n_validation": len(val),
                         "elapsed_seconds": time.perf_counter() - start,
                         "model_path": str(model_path.relative_to(PROJECT_DIR))})
    # The RF candidate is the ensemble of all frozen seeds, averaged in log space.
    for partition, data in [("train", train), ("validation", val)]:
        ensemble_log = np.mean(forest_log_predictions[partition], axis=0)
        prediction_blocks.append(prediction_frame(data, ensemble_log, "random_forest", split_seed, fold, partition))
    # Incremental checkpoints; interruption does not erase completed folds.
    pd.concat(prediction_blocks, ignore_index=True).to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    pd.concat(seed_prediction_blocks, ignore_index=True).to_csv(OUTPUT_DIR / "forest_seed_predictions.csv", index=False)
    pd.DataFrame(fit_logs).to_csv(OUTPUT_DIR / "fit_log.csv", index=False)
    print(f"Completed split {split_seed}, fold {fold}")

predictions = pd.concat(prediction_blocks, ignore_index=True)
assert len(fit_logs) == len(folds) * (2 + len(protocol["rf_seeds"]))
validation = predictions.query("partition == 'validation'").copy()
assert not set(validation.track_id) & holdout_ids
for (name, seed), data in validation.groupby(["model", "split_seed"]):
    assert len(data) == 68 and data.track_id.nunique() == 68
    assert set(data.track_id) == set(dev.index)
print("All development tracks have exactly one out-of-fold prediction per repetition and candidate.")

## Metrics and diagnostics

In [ ]:
def score_frame(data):
    abs_log = np.abs(data.y_true_log - data.y_pred_log)
    group_errors = pd.DataFrame({"album_family": data.album_family, "error": abs_log})
    return {
        "mae_log_macro": group_errors.groupby("album_family").error.mean().mean(),
        "mae_log": mean_absolute_error(data.y_true_log, data.y_pred_log),
        "rmse_log": np.sqrt(mean_squared_error(data.y_true_log, data.y_pred_log)),
        "r2_log": r2_score(data.y_true_log, data.y_pred_log),
        "mae_streams": mean_absolute_error(data.y_true, data.y_pred),
        "rmse_streams": np.sqrt(mean_squared_error(data.y_true, data.y_pred)),
        "r2_streams": r2_score(data.y_true, data.y_pred)
    }

# R² is calculated over all 68 OOF predictions per repetition, not averaged over folds.
metric_rows = []
for (model_name, split_seed, partition), data in predictions.groupby(["model", "split_seed", "partition"]):
    metric_rows.append({"model": model_name, "split_seed": split_seed, "partition": partition, **score_frame(data)})
metrics = pd.DataFrame(metric_rows)
assert np.isfinite(metrics.select_dtypes(include="number")).all().all()
metrics.to_csv(OUTPUT_DIR / "metrics_by_repeat.csv", index=False)

family_errors = validation.assign(absolute_log_error=lambda x: abs(x.y_true_log - x.y_pred_log)).groupby(
    ["model", "split_seed", "album_family"], as_index=False
).agg(mae_log=("absolute_log_error", "mean"), n_tracks=("track_id", "size"))
family_errors.to_csv(OUTPUT_DIR / "validation_errors_by_family.csv", index=False)
val_metrics = metrics.query("partition == 'validation'")
summary = val_metrics.groupby("model").agg(
    mae_log_macro_mean=("mae_log_macro", "mean"),
    mae_log_macro_sd_between_repeats=("mae_log_macro", "std"),
    mae_streams_mean=("mae_streams", "mean"),
    rmse_streams_mean=("rmse_streams", "mean"),
    r2_streams_mean=("r2_streams", "mean"),
    r2_log_mean=("r2_log", "mean")
).sort_values("mae_log_macro_mean")
summary.to_csv(OUTPUT_DIR / "validation_summary.csv")
print("VALIDATION SUMMARY — no final model selection yet")
print(summary.round(4).to_string())
print("\nTRAINING VS VALIDATION — primary metric")
gap = metrics.groupby(["model", "partition"]).mae_log_macro.mean().unstack()
gap["validation_minus_train"] = gap["validation"] - gap["train"]
gap.to_csv(OUTPUT_DIR / "training_validation_gap.csv")
print(gap.round(4).to_string())
print("Repeat SD is descriptive, not a standard error or confidence interval.")

## Investigating extreme predictions
A finite prediction can still be implausible. Inspect the largest linear validation prediction and how far its inputs lie from that fold’s training distribution. This is a diagnostic, not permission to alter the frozen experiment.

In [ ]:
# Investigate extreme original-scale predictions without changing the protocol.
linear_val = validation.loc[validation.model.eq("linear")]
extreme = linear_val.loc[linear_val.y_pred.idxmax()]
extreme_estimator = joblib.load(MODEL_DIR / f"split_{int(extreme.split_seed)}_fold_{int(extreme.fold)}_linear.joblib")
extreme_x = dev.loc[[int(extreme.track_id)], FEATURES]
standardized = extreme_estimator.regressor_[:-1].transform(extreme_x)[0]
extreme_features = pd.DataFrame({"feature": FEATURES, "standardized_value": standardized})
extreme_features["absolute_standardized_value"] = abs(extreme_features.standardized_value)
extreme_features = extreme_features.sort_values("absolute_standardized_value", ascending=False)
extreme_features.to_csv(OUTPUT_DIR / "linear_extreme_prediction_features.csv", index=False)
print("Largest linear validation prediction:")
print(extreme[["track_id", "album_family", "split_seed", "fold", "y_true", "y_pred", "y_pred_log"]].to_string())
print("Most extreme standardized inputs relative to that fold's training data:")
print(extreme_features.head().to_string(index=False))
print("Do not clip predictions or remove tracks to improve scores after seeing this. Document the instability.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
order = summary.index.tolist()
for i, name in enumerate(order):
    values = val_metrics.loc[val_metrics.model.eq(name), "mae_log_macro"].to_numpy()
    axes[0].scatter(np.full(len(values), i), values, alpha=0.7, label=name)
    axes[0].plot([i-0.2, i+0.2], [values.mean()]*2, color="black", linewidth=2)
axes[0].set_xticks(range(len(order)), order)
axes[0].set(ylabel="Macro MAE on log1p streams (lower is better)", title="Validation: five repeated group splits")
gap.loc[order, ["train", "validation"]].plot.bar(ax=axes[1], rot=0, color=["#9bbbd1", "#315c80"])
axes[1].set(ylabel="Macro MAE on log1p streams", xlabel="", title="Training vs validation")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "validation_and_overfitting.png", dpi=180)
plt.show()

family_plot = family_errors.groupby(["album_family", "model"]).mae_log.mean().unstack().reindex(columns=order)
fig, ax = plt.subplots(figsize=(10, 5))
family_plot.plot.barh(ax=ax)
ax.set(xlabel="Mean validation MAE on log1p streams", ylabel="", title="Performance varies across album families")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "validation_by_album_family.png", dpi=180)
plt.show()

In [ ]:
metadata = {
    "protocol_version": protocol["version"], "source_sha256": protocol["source_sha256"],
    "input_protocol_path": str(protocol_path.relative_to(PROJECT_DIR)),
    "input_folds_path": str(fold_path.relative_to(PROJECT_DIR)),
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "joblib": joblib.__version__,
    "development_tracks": len(dev), "n_folds": len(folds), "n_fitted_estimators": len(fit_logs),
    "split_seeds": protocol["split_seeds"], "rf_seeds": protocol["rf_seeds"],
    "rf_aggregation": "mean of log predictions, followed by expm1",
    "final_test_evaluated": False, "model_selected": False,
    "training_metrics_note": "Each track appears in two training folds per repeat; training scores are diagnostic, not generalization estimates."
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "frozen_protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "frozen_folds.json").write_text(json.dumps(folds, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "requirements_snapshot.txt").write_text(
    f"pandas=={pd.__version__}\nnumpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\n", encoding="utf-8")
print("75 estimators fitted and saved; 3 candidate procedures evaluated.")
print("Final holdout not evaluated. Next: paired family-level statistical analysis and selection.")

## Interpretation rules
- Smaller MAE/RMSE is better. The main comparison gives equal weight to the six development families.
- Original-scale MAE and RMSE are in accumulated streams; log metrics use the natural logarithm.
- Negative R² is a valid possible result: squared prediction error exceeds the reference based on the evaluation-set mean. This reference is not the same as our training-log-mean baseline.
- The forest is one candidate procedure using three seeds, not three competing candidates. Never pick the best seed.
- Training performance better than validation suggests optimism; it does not alone distinguish overfitting from distribution shifts across album families.
- Small variation between repeated splits does not establish population-level certainty. Only six development families exist.
- Baseline and linear have no model-seed randomness. Their variability here comes from group partitions.
- Saved joblib files are fold-specific development models, not a final deliverable model. Load only model files you trust and use matching package versions.

## Completion criterion
All 75 fits finish, assertions pass, and every candidate has 68 OOF predictions per repetition. Save the entire output directory; it contains intermediate models, predictions, logs, plots and the frozen protocol.

Next stage: paired differences by album family and exploratory cluster bootstrap as specified in the protocol. Only then freeze the candidate and evaluate the untouched-since-reservation final holdout. Do not change settings after inspecting final test results.